In [ ]:
# Jak to spoustim?

# https://www.idiap.ch/software/bob/docs/bob/docs/stable/install.html
# mamba create --name bob --override-channels   -c https://www.idiap.ch/software/bob/conda   -c conda-forge   python=3.10 bob.bio.vein ipykernel opencv
# mamba activate bob

# Vytvoreni kernelu
# python -m ipykernel install --user --name=bob --display-name "bob.bio.vein kernel

# Potom mamba activate base a odtud spoustim python notebook:
# jupyter notebook --no-browser --ip=0.0.0.0 --port=8888

# napravo nahore vyberu kernel
# hotovo

# !!!!!! Pokud to pise problem s scikit learn safe_tags, staci spustit bunku znovu, deje se to jen po nacteni kernelu !!!!!!

In [ ]:
import zipfile
from pathlib import Path

# Cesta k zip archivu s výstupem skriptu process_images.sh
zip_path = Path("original_processed.zip")
extract_to = Path("comparison_data")

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_to)

print("Archiv rozbalen do:", extract_to.resolve())

In [ ]:
# Roztřídění obrázků do příslušných složek

import os
from pathlib import Path
import shutil
from PIL import Image
import numpy as np

# Vstupní složka (s extrahovaným výstupem skriptu process_images.sh
input_dir = Path("comparison_data/original_processed")

# Výstupní složky
output_dirs = {
    "original": Path("comparison_data/orig"), # Původní obrázky
    "enhanced": Path("comparison_data/enhanced"), # Enhanced obrázky
    "baseline_masks": Path("comparison_data/baseline"), # Zarovnané baseline masky
}
for path in output_dirs.values():
    path.mkdir(parents=True, exist_ok=True)

# Automatické zarovnání a ořez baseline masek podle enhanced obrázku
def auto_align_mask_array(ref_img: Image.Image, mask_img: Image.Image, max_shift=15):
    
    # Převod na grayscale
    ref = np.array(ref_img.convert("L"), dtype=np.float32)
    mask = np.array(mask_img.convert("L"), dtype=np.float32)

    # Oříznutí masky na velikost referenčního obrázku
    h_ref, w_ref = ref.shape
    mask_bin = mask[:h_ref, :w_ref] / 255.0

    # Invertování referenčního obrázku
    ref_inv = 255 - ref
    ref_inv /= 255.0

    # Iniciální nastavení
    best_score = -1
    best_dx, best_dy = 0, 0

    # Hledání nejlepšího posunu v daném okolí
    for dy in range(-max_shift, max_shift + 1):
        for dx in range(-max_shift, max_shift + 1):
            shifted = np.roll(mask_bin, shift=(dy, dx), axis=(0, 1))
            score = np.sum(shifted * ref_inv)
            if score > best_score:
                best_score = score
                best_dx, best_dy = dx, dy

    # Aplikace nejlepšího posunu
    aligned_mask = np.roll(mask_bin, shift=(best_dy, best_dx), axis=(0, 1))
    aligned_mask_img = Image.fromarray((aligned_mask * 255).astype(np.uint8))
    return aligned_mask_img, best_dx, best_dy


# Zpracování souborů - všechny jsou jpg
for file in input_dir.glob("*.jpg"):
    name = file.stem

    # Baseline masky
    if name.endswith("_enhanced_mc"):
        base_name = name.replace("_enhanced_mc", "")
        dest = output_dirs["baseline_masks"] / f"{base_name}.jpg"

        enh_path = input_dir / f"{base_name}_enhanced.jpg"

        enhanced_img = Image.open(enh_path).convert("RGB")
        mc = Image.open(file).convert("RGB")

        aligned_mask, dx, dy = auto_align_mask_array(enhanced_img, mc)
        aligned_mask.save(dest)
        print(f"Maska {base_name} zarovnána podle enhanced (dx={dx}, dy={dy})")

    # Enhanced obrázky
    elif name.endswith("_enhanced"):
        base_name = name.replace("_enhanced", "")
        dest = output_dirs["enhanced"] / f"{base_name}.jpg"
        shutil.copy2(file, dest)

    # Původní obrázky
    else:
        base_name = name
        dest = output_dirs["original"] / f"{base_name}.jpg"
        shutil.copy2(file, dest)

print("Obrázky roztříděny a masky zarovnány.")


In [ ]:
import os
import numpy as np
from PIL import Image
from skimage.filters import threshold_otsu
import cv2
from bob.bio.vein.extractor import MaximumCurvature
import pandas as pd
import matplotlib.pyplot as plt

NUM_IMAGES = 500 # Kolik obrázků z každé metody zpracovat
START_INDEX = 0 # Jakým obrázkem začít (index)

BASE_DIR = "comparison_data" # Kde jsou data

# Jednotlivé metody
METHOD_DIRS = [
    "original",
    "fixed_gabor_filters",
    "adaptive_gabor_filters",
    "adaptive_gabor_filters_more",
]

# Kde jsou baseline masky
BASELINE_DIR = os.path.join(BASE_DIR, "baseline")

# Zda zobrazovat vizualizaci pro každý obrázek
SHOW_VISUALS = False

mc_extractor = MaximumCurvature(sigma=5.5)

# Aplikuje MC, podle Otsu threshold binarizuje
def apply_maximum_curvature(img, mask):
    features = mc_extractor((img.astype(np.float32), mask))
    thresh = threshold_otsu(features)
    binary = (features > thresh).astype(np.uint8)
    return binary

# Vytvoří masku prstu, případně ji vyplní a vyhladí
def create_finger_mask_lee(image, window_size=10, noise_var=0.03, morph_kernel_size=5):
    img = image.astype(np.float32)
    local_mean = cv2.blur(img, (window_size, window_size))
    local_sqr_mean = cv2.blur(img**2, (window_size, window_size))
    local_var = local_sqr_mean - local_mean**2
    gain = local_var / (local_var + noise_var)
    adaptive_img = local_mean + gain * (img - local_mean)
    mask = adaptive_img > np.mean(adaptive_img)
    mask_filled = mask.copy()
    h, w = mask.shape
    for x in range(w):
        column = mask[:, x]
        ys = np.where(column)[0]
        if len(ys) > 0:
            y_top, y_bottom = ys[0], ys[-1]
            mask_filled[y_top:y_bottom+1, x] = True
    kernel = np.ones((morph_kernel_size, morph_kernel_size), np.uint8)
    mask_closed = cv2.morphologyEx(mask_filled.astype(np.uint8), cv2.MORPH_CLOSE, kernel)
    mask_smooth = cv2.dilate(mask_closed, kernel, iterations=1)
    return mask_smooth.astype(bool)

# Vypočte metriky TP, FP, FN, TN
def compute_confusion_metrics(pred, gt):
    if pred.shape != gt.shape:
        pred = cv2.resize(pred.astype(np.uint8), (gt.shape[1], gt.shape[0]), interpolation=cv2.INTER_NEAREST)
    pred = pred.astype(bool)
    gt = gt.astype(bool)
    TP = np.sum(pred & gt)
    FP = np.sum(pred & ~gt)
    FN = np.sum(~pred & gt)
    TN = np.sum(~pred & ~gt)
    total = TP + FP + FN + TN
    return TP, FP, FN, TN

# Vizualizace pro jeden snímek
def visualize_full_comparison(image, finger_mask, pred_bin, gt_bin, title):
    if pred_bin.shape != gt_bin.shape:
        pred_bin = cv2.resize(pred_bin.astype(np.uint8), (gt_bin.shape[1], gt_bin.shape[0]), interpolation=cv2.INTER_NEAREST)
    if finger_mask.shape != image.shape:
        finger_mask = cv2.resize(finger_mask.astype(np.uint8), (image.shape[1], image.shape[0]), interpolation=cv2.INTER_NEAREST)

    image_norm = (image - image.min()) / (image.max() - image.min())
    image_rgb = np.stack([image_norm]*3, axis=-1)
    finger_mask_rgb = np.stack([finger_mask.astype(np.uint8)*255]*3, axis=-1)
    combined_overlay = np.zeros_like(image_rgb)
    combined_overlay[..., 0] = pred_bin
    combined_overlay[..., 2] = gt_bin
    gt_overlay = image_rgb.copy()
    gt_overlay[..., 2] = np.maximum(gt_overlay[..., 2], gt_bin)
    pred_overlay = image_rgb.copy()
    pred_overlay[..., 0] = np.maximum(pred_overlay[..., 0], pred_bin)

    fig, axs = plt.subplots(1, 5, figsize=(22, 5))
    axs[0].imshow(image, cmap="gray"); axs[0].set_title("Původní obraz prstu"); axs[0].axis("off")
    axs[1].imshow(finger_mask_rgb, cmap="gray"); axs[1].set_title("Finger mask"); axs[1].axis("off")
    axs[2].imshow(combined_overlay); axs[2].set_title("Predikce + GT (fialová = shoda)"); axs[2].axis("off")
    axs[3].imshow(gt_overlay); axs[3].set_title("Ground truth (modře)"); axs[3].axis("off")
    axs[4].imshow(pred_overlay); axs[4].set_title("Predikce (červeně)"); axs[4].axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

# Zpracování jednoho snímku
def process_image(image_path, baseline_path, mask_lee, show_visual=True, baseline_dilate_size=4):
    img = Image.open(image_path).convert("L")
    image = np.array(img, dtype=np.float32)

    baseline = Image.open(baseline_path).convert("L")
    baseline = np.array(baseline, dtype=np.uint8)
    baseline_bin = (baseline > 127).astype(np.uint8)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (baseline_dilate_size, baseline_dilate_size))
    baseline_bin_dilated = cv2.dilate(baseline_bin, kernel, iterations=1)

    binary = apply_maximum_curvature(image, mask_lee)
    pred_bin = (binary > 0).astype(np.uint8)
    TP, FP, FN, TN = compute_confusion_metrics(pred_bin, baseline_bin_dilated)

    if show_visual:
        title = os.path.basename(image_path)
        visualize_full_comparison(image, mask_lee, pred_bin, baseline_bin_dilated, title)

    return {"TP": TP, "FP": FP, "FN": FN, "TN": TN}

# Main

finger_masks = {}
results = []

enhanced_dir = os.path.join(BASE_DIR, "enhanced")
enhanced_files = [f for f in os.listdir(enhanced_dir) if f.lower().endswith(".jpg")]
enhanced_files.sort()
selected_files = enhanced_files[START_INDEX : START_INDEX + NUM_IMAGES]

# Předpočítání masek podle enhanced obrázku prstu - nejspolehlivější
for f in selected_files:
    image_path = os.path.join(enhanced_dir, f)
    img = Image.open(image_path).convert("L")
    image = np.array(img, dtype=np.float32)
    mask_lee = create_finger_mask_lee(image)
    finger_masks[f] = mask_lee

for method in METHOD_DIRS:
    print(f"\n=== Zpracovávám metodu: {method} ===")
    method_dir = os.path.join(BASE_DIR, method)
    files = [f for f in os.listdir(method_dir) if f.lower().endswith(".jpg")]
    files.sort()
    selected_files = files[START_INDEX : START_INDEX + NUM_IMAGES]

    for f in selected_files:
        image_path = os.path.join(method_dir, f)
        baseline_path = os.path.join(BASELINE_DIR, f)

        mask_lee = finger_masks.get(f)

        if mask_lee is None:
            img = Image.open(image_path).convert("L")
            image = np.array(img, dtype=np.float32)
            mask_lee = create_finger_mask_lee(image)

        metrics = process_image(image_path, baseline_path, mask_lee, show_visual=SHOW_VISUALS)
        results.append({
            "file": f,
            "method": method,
            **metrics
        })
        print(f"{f} [{method}] -> TP={metrics['TP']}, FP={metrics['FP']}, FN={metrics['FN']}, TN={metrics['TN']}")

df = pd.DataFrame(results)
df.to_csv("results_confusion_metrics.csv", index=False)
print("\nVýsledky uložené do 'results_confusion_metrics.csv'")


In [ ]:
# Vyhodnocení metrik

import pandas as pd

# Soubor s vypočtenými metrikami pro každý obrázek a každou metodu
df = pd.read_csv("results_confusion_metrics.csv")

# Vynechání chybných měření - v malém procentu případů automatické zarovnání selže
# Můžeme bezpečně ignorovat, protože pro každé "selhání" vypustíme příslušný snímek z každé metody
df_clean = df[df["FP"] <= df["TP"]].copy()

# Průměrné hodnoty pro každou metodu
mean_metrics = df_clean.groupby("method")[["TP", "FP", "FN", "TN"]].mean()

# Souhrnné hodnoty (sumy přes všechny obrázky)
sum_metrics = df_clean.groupby("method")[["TP", "FP", "FN", "TN"]].sum()

# Odvozené metriky - accuracy, precision, recall a F1-score
sum_metrics["Accuracy"] = (sum_metrics["TP"] + sum_metrics["TN"]) / (sum_metrics["TP"] + sum_metrics["FP"] + sum_metrics["FN"] + sum_metrics["TN"])
sum_metrics["Precision"] = sum_metrics["TP"] / (sum_metrics["TP"] + sum_metrics["FP"])
sum_metrics["Recall"] = sum_metrics["TP"] / (sum_metrics["TP"] + sum_metrics["FN"])
sum_metrics["F1"] = 2 * (sum_metrics["Precision"] * sum_metrics["Recall"]) / (sum_metrics["Precision"] + sum_metrics["Recall"])

# Výstup
print("=== Průměrné hodnoty (po odfiltrování chybných měření) ===")
print(mean_metrics.round(2))
print("\n=== Souhrnné metriky ===")
print(sum_metrics[["Accuracy", "Precision", "Recall", "F1"]].round(4))
